# Early churn model for a medical scribe product

**Question:** which doctors and institutions are more prone to early churn, and how early can we tell?

**Data (synthetic).** Every row in this repository is simulated by `make_synthetic_data.py`; no real doctors or institutions are involved, and the numbers below are illustrative only.
- `data/cohort_roster.csv`: one row per doctor, retained and churned, with first-week usage.
- `data/weekly_usage.csv`: consultations and active days per week for each doctor's first four weeks.

**Contents**
- **Part A — Monthly churn.** Churned = no consultation in the first 20 days of the month after the first session.
- **Part B — 4-week retention.** Retained = at least one consultation on days 22–28 after the first session.
- **Summary** of what both parts show.

Run the notebook top to bottom; later cells depend on earlier ones.

## 0. Setup

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score

# the same 5 folds are used for every cross-validation, so scores are comparable
folds = StratifiedKFold(5, shuffle=True, random_state=0)

---
# Part A — Monthly churn

## A1. Load the data

`churned` is 1 when the doctor had no consultation in the first 20 days of the following month (21 for June).

In [2]:
df = pd.read_csv('data/cohort_roster.csv', parse_dates=['cohort_month', 'first_session_date'])

print(df.shape)
df.groupby('cohort_month')['churned'].agg(['size', 'mean']).round(3)

(2140, 9)


,size,mean
cohort_month,,
2026-06-01,757,0.346
2026-07-01,828,0.327
2026-08-01,555,0.310


## A2. Keep doctors with a full first week

A doctor who starts in the last days of the month has a "first week" that runs into the period used to decide churn. Using it would leak the answer into the inputs, so we keep only doctors with at least 7 days left in their first month.

In [3]:
month_end = df['cohort_month'] + pd.offsets.MonthEnd(0)
df['days_available'] = (month_end - df['first_session_date']).dt.days + 1
df['eligible_7d'] = df['days_available'] >= 7

eligible = df[df['eligible_7d']]
print(len(eligible), 'of', len(df), 'doctors have a full first week')

1715 of 2140 doctors have a full first week


## A3. Keep institutions with at least 20 doctors

Applied after the first-week filter, because an institution can drop below 20 once late starters are removed.

In [4]:
counts = eligible['institutional_name'].value_counts()
big = counts[counts >= 20].index

model_df = eligible[eligible['institutional_name'].isin(big)].copy()

print(len(big), 'institutions,', len(model_df), 'doctors,', model_df['churned'].sum(), 'churned')

16 institutions, 1662 doctors, 576 churned


## A4. Turn institution and specialty into numbers

Each institution gets its own 0/1 column (one-hot encoding), so the model can learn a separate effect for each. Specialty becomes a single flag, since most doctors are general medicine.

In [5]:
model_df['is_general_medicine'] = (model_df['primary_specialty'] == 'generalMedicine').astype(int)

X_inst = pd.get_dummies(model_df['institutional_name'], prefix='inst', dtype=int)
inst_cols = list(X_inst.columns)

print(X_inst.shape)
X_inst.sum().sort_values(ascending=False)

(1662, 16)


inst_Institution A    571
inst_Institution B    154
inst_Institution C    142
inst_Institution D    140
inst_Institution E     95
inst_Institution F     84
inst_Institution G     82
inst_Institution H     75
inst_Institution I     71
inst_Institution J     53
inst_Institution K     45
inst_Institution L     42
inst_Institution M     30
inst_Institution N     29
inst_Institution O     27
inst_Institution P     22
dtype: int64

## A5. Build the model inputs

`X` holds the inputs: institution columns, the specialty flag, and three first-week usage numbers (logged, because counts are very lopsided). `y` is the answer.

In [6]:
usage_cols = ['sessions_day1', 'sessions_first7d', 'active_days_first7d']

X = pd.concat([X_inst, model_df[['is_general_medicine']], np.log1p(model_df[usage_cols])], axis=1)
y = model_df['churned']

X.shape

(1662, 20)

## A6. Honest test: train on June–July, predict August

The scaler is fitted on the training months only, then applied to August. AUC is the chance the model ranks a random churner above a random retained doctor (0.5 = guessing, 1.0 = perfect).

In [7]:
is_test = model_df['cohort_month'] == '2026-08-01'
X_train, y_train = X[~is_test].copy(), y[~is_test]
X_test, y_test = X[is_test].copy(), y[is_test]

scaler = StandardScaler()
X_train[usage_cols] = scaler.fit_transform(X_train[usage_cols])
X_test[usage_cols] = scaler.transform(X_test[usage_cols])

model = LogisticRegression(C=1.0, max_iter=1000)
model.fit(X_train, y_train)

p_test = model.predict_proba(X_test)[:, 1]
print(X_train.shape, X_test.shape)
print('AUC on August:', round(roc_auc_score(y_test, p_test), 3))

(1226, 20) (436, 20)
AUC on August: 0.854


## A7. Final model: fit on all three months

Institutions onboard in waves, so June–July alone has almost no doctors for some of them. To compare institutions we fit on everyone and measure accuracy by cross-validation instead.

Coefficients: negative means lower churn, positive means higher churn.

In [8]:
X_all = X.copy()
X_all[usage_cols] = StandardScaler().fit_transform(X_all[usage_cols])

auc = cross_val_score(LogisticRegression(max_iter=1000), X_all, y, cv=folds, scoring='roc_auc')
print('Cross-validated AUC:', round(auc.mean(), 3))

final_model = LogisticRegression(C=1.0, max_iter=1000).fit(X_all, y)

pd.Series(final_model.coef_[0], index=X_all.columns).sort_values().round(2)

Cross-validated AUC: 0.849


inst_Institution G    -1.21
active_days_first7d   -0.97
inst_Institution I    -0.61
sessions_first7d      -0.55
inst_Institution J    -0.49
inst_Institution A    -0.46
inst_Institution E    -0.40
inst_Institution D    -0.33
inst_Institution C    -0.26
inst_Institution N    -0.21
inst_Institution L    -0.18
is_general_medicine   -0.06
sessions_day1         -0.03
inst_Institution K    -0.01
inst_Institution M     0.10
inst_Institution H     0.39
inst_Institution B     0.58
inst_Institution P     0.86
inst_Institution F     0.92
inst_Institution O     0.94
dtype: float64

## A8. Adjusted churn per institution

For each institution, pretend all doctors worked there (keeping their real first-week usage and specialty) and average the predicted churn. This compares institutions on the same mix of doctors.

In [9]:
rows = []
for col in inst_cols:
    X_what_if = X_all.copy()      # everyone keeps their real usage and specialty
    X_what_if[inst_cols] = 0      # remove every doctor from their institution
    X_what_if[col] = 1            # and place them all in this one
    rows.append({'institution': col[5:], 'adjusted_churn': final_model.predict_proba(X_what_if)[:, 1].mean()})

raw = model_df.groupby('institutional_name')['churned'].agg(doctors='size', raw_churn='mean')
result = raw.join(pd.DataFrame(rows).set_index('institution')).sort_values('adjusted_churn', ascending=False)
result.round(3)

,doctors,raw_churn,adjusted_churn
institutional_name,,,
Institution O,27,0.815,0.528
Institution F,84,0.810,0.525
Institution P,22,0.682,0.516
Institution B,154,0.539,0.469
Institution H,75,0.307,0.436
Institution M,30,0.367,0.390
Institution K,45,0.356,0.372
Institution L,42,0.571,0.346
Institution N,29,0.276,0.341


## A9. Churn by first-week usage

No model here, just a count per usage group.

In [10]:
buckets = pd.cut(model_df['sessions_first7d'], [0, 3, 10, 30, 1000], labels=['1-3', '4-10', '11-30', '31+'])

model_df.groupby(buckets, observed=True)['churned'].agg(doctors='size', churn='mean').round(3)

,doctors,churn
sessions_first7d,,
1-3,405,0.743
4-10,509,0.389
11-30,515,0.122
31+,233,0.060


## A10. Decision tree: simple rules

A tree asks up to three yes/no questions and sorts doctors into at most 8 groups. It uses the raw counts, so the thresholds read directly as numbers of consultations. In the printout, `inst_X <= 0.50` means "not at X".

In [11]:
X_tree = pd.concat([X_inst, model_df[['is_general_medicine']], model_df[usage_cols]], axis=1)

tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=0)
auc_tree = cross_val_score(tree, X_tree, y, cv=folds, scoring='roc_auc')
print('Tree cross-validated AUC:', round(auc_tree.mean(), 3))

tree.fit(X_tree, y)
print(export_text(tree, feature_names=list(X_tree.columns)))

Tree cross-validated AUC: 0.832
|--- active_days_first7d <= 2.50
|   |--- active_days_first7d <= 1.50
|   |   |--- inst_Institution A <= 0.50
|   |   |   |--- class: 1
|   |   |--- inst_Institution A >  0.50
|   |   |   |--- class: 1
|   |--- active_days_first7d >  1.50
|   |   |--- sessions_first7d <= 5.50
|   |   |   |--- class: 1
|   |   |--- sessions_first7d >  5.50
|   |   |   |--- class: 0
|--- active_days_first7d >  2.50
|   |--- sessions_first7d <= 10.50
|   |   |--- sessions_first7d <= 6.50
|   |   |   |--- class: 0
|   |   |--- sessions_first7d >  6.50
|   |   |   |--- class: 0
|   |--- sessions_first7d >  10.50
|   |   |--- sessions_day1 <= 2.50
|   |   |   |--- class: 0
|   |   |--- sessions_day1 >  2.50
|   |   |   |--- class: 0



Churn rate in each of the tree's final groups (`leaf` is just an internal label):

In [12]:
model_df['leaf'] = tree.apply(X_tree)

leaves = model_df.groupby('leaf').agg(
    doctors=('churned', 'size'),
    churn=('churned', 'mean'),
    min_sessions=('sessions_first7d', 'min'),
    max_sessions=('sessions_first7d', 'max'),
)
leaves.sort_values('churn', ascending=False).round(3)

,doctors,churn,min_sessions,max_sessions
leaf,,,,
3,276,0.826,1,12
4,94,0.713,1,12
6,177,0.548,2,5
10,73,0.397,3,6
7,125,0.376,6,22
11,192,0.214,7,10
13,84,0.190,11,32
14,641,0.080,11,188


---
# Part B — 4-week retention

Same approach, different outcome. Week 4 is counted from each doctor's own first session, so no doctor is excluded for starting late in the month. Variables in this part end in `4` so they don't overwrite Part A.

## B1. Load weekly usage and build the label

`a_w4` is the number of active days in week 4 (days 22–28). `lost_w4` is 1 when the doctor had no consultation that week; the models predict this, so a higher number still means "worse".

In [13]:
weekly = pd.read_csv('data/weekly_usage.csv')
df4 = df.merge(weekly, on='account_id')

df4['retained_w4'] = (df4['a_w4'] > 0).astype(int)
df4['lost_w4'] = 1 - df4['retained_w4']

print(len(df4), 'doctors, retained at week 4:', round(df4['retained_w4'].mean(), 3))
df4.groupby('cohort_month')['retained_w4'].agg(['size', 'mean']).round(3)

2140 doctors, retained at week 4: 0.616


,size,mean
cohort_month,,
2026-06-01,757,0.588
2026-07-01,828,0.627
2026-08-01,555,0.640


Share of doctors active in each of their first four weeks:

In [14]:
pd.Series({f'week {k}': (df4[f'a_w{k}'] > 0).mean() for k in (1, 2, 3, 4)}).round(3)

week 1    1.000
week 2    0.666
week 3    0.631
week 4    0.616
dtype: float64

## B2. Institution filter and encoding

In [15]:
counts4 = df4['institutional_name'].value_counts()
big4 = counts4[counts4 >= 20].index
model4 = df4[df4['institutional_name'].isin(big4)].copy()

model4['is_general_medicine'] = (model4['primary_specialty'] == 'generalMedicine').astype(int)
X_inst4 = pd.get_dummies(model4['institutional_name'], prefix='inst', dtype=int)
inst_cols4 = list(X_inst4.columns)

print(len(big4), 'institutions,', len(model4), 'doctors,', model4['lost_w4'].sum(), 'not retained')

17 institutions, 2097 doctors, 814 not retained


## B3. Logistic regression

Inputs are week-1 numbers only (`s_w1` consultations, `a_w1` active days). Weeks 2–4 stay out, since they are what we are predicting.

In [16]:
usage_cols4 = ['sessions_day1', 's_w1', 'a_w1']

X4 = pd.concat([X_inst4, model4[['is_general_medicine']], np.log1p(model4[usage_cols4])], axis=1)
X4[usage_cols4] = StandardScaler().fit_transform(X4[usage_cols4])
y4 = model4['lost_w4']

auc4 = cross_val_score(LogisticRegression(max_iter=1000), X4, y4, cv=folds, scoring='roc_auc')
print('Cross-validated AUC:', round(auc4.mean(), 3))

model_w4 = LogisticRegression(C=1.0, max_iter=1000).fit(X4, y4)
pd.Series(model_w4.coef_[0], index=X4.columns).sort_values().round(2)

Cross-validated AUC: 0.854


a_w1                  -0.95
inst_Institution G    -0.89
inst_Institution J    -0.67
s_w1                  -0.60
inst_Institution A    -0.51
inst_Institution D    -0.46
inst_Institution I    -0.31
inst_Institution E    -0.26
inst_Institution N    -0.26
inst_Institution C    -0.22
inst_Institution H    -0.16
inst_Institution M    -0.13
sessions_day1         -0.06
is_general_medicine    0.07
inst_Institution K     0.08
inst_Institution Q     0.10
inst_Institution L     0.16
inst_Institution B     0.50
inst_Institution O     0.85
inst_Institution P     0.91
inst_Institution F     0.99
dtype: float64

## B4. Adjusted share not retained, per institution

In [17]:
rows4 = []
for col in inst_cols4:
    X_what_if = X4.copy()
    X_what_if[inst_cols4] = 0
    X_what_if[col] = 1
    rows4.append({'institution': col[5:], 'adjusted_lost': model_w4.predict_proba(X_what_if)[:, 1].mean()})

raw4 = model4.groupby('institutional_name')['lost_w4'].agg(doctors='size', raw_lost='mean')
result4 = raw4.join(pd.DataFrame(rows4).set_index('institution')).sort_values('adjusted_lost', ascending=False)
result4.round(3)

,doctors,raw_lost,adjusted_lost
institutional_name,,,
Institution F,110,0.855,0.582
Institution P,25,0.720,0.569
Institution O,30,0.833,0.559
Institution B,200,0.565,0.501
Institution L,50,0.700,0.446
Institution Q,22,0.545,0.435
Institution K,60,0.433,0.432
Institution M,40,0.325,0.398
Institution H,95,0.284,0.394


## B5. Decision tree

In [18]:
X_tree4 = pd.concat([X_inst4, model4[['is_general_medicine']], model4[usage_cols4]], axis=1)

tree4 = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=0)
auc_tree4 = cross_val_score(tree4, X_tree4, y4, cv=folds, scoring='roc_auc')
print('Tree cross-validated AUC:', round(auc_tree4.mean(), 3))

tree4.fit(X_tree4, y4)
print(export_text(tree4, feature_names=list(X_tree4.columns)))

Tree cross-validated AUC: 0.843
|--- s_w1 <= 5.50
|   |--- a_w1 <= 1.50
|   |   |--- inst_Institution A <= 0.50
|   |   |   |--- class: 1
|   |   |--- inst_Institution A >  0.50
|   |   |   |--- class: 1
|   |--- a_w1 >  1.50
|   |   |--- inst_Institution F <= 0.50
|   |   |   |--- class: 1
|   |   |--- inst_Institution F >  0.50
|   |   |   |--- class: 1
|--- s_w1 >  5.50
|   |--- s_w1 <= 11.50
|   |   |--- a_w1 <= 2.50
|   |   |   |--- class: 0
|   |   |--- a_w1 >  2.50
|   |   |   |--- class: 0
|   |--- s_w1 >  11.50
|   |   |--- a_w1 <= 3.50
|   |   |   |--- class: 0
|   |   |--- a_w1 >  3.50
|   |   |   |--- class: 0



In [19]:
model4['leaf'] = tree4.apply(X_tree4)

model4.groupby('leaf').agg(
    doctors=('lost_w4', 'size'),
    lost=('lost_w4', 'mean'),
).sort_values('lost', ascending=False).round(3)

,doctors,lost
leaf,,
7,31,0.935
3,342,0.880
4,113,0.752
6,255,0.565
10,160,0.462
11,332,0.277
13,162,0.179
14,702,0.085


---
# Summary

What this analysis is designed to show (the figures above come from synthetic data):

- **How much first-week usage predicts churn**, compared with institution and specialty (sections A7 and A9).
- **Which institutions differ once usage is accounted for**, using the adjusted rate (A8 and B4). Small differences in the middle of the table should not be read as a ranking.
- **Simple rules a team can act on at day 7**, from the decision trees (A10 and B5).
- **Association, not cause.** A doctor who was already leaving also uses the tool little, so higher usage would not necessarily have retained them.